# Volcano Plot Development

Interactive development of volcano plots for differential metabolite analysis.

**Purpose:**
- Visualize fold changes vs statistical significance
- Identify significantly differential metabolites
- Customize plot aesthetics
- Export publication-ready figures

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# For high-resolution plots
%config InlineBackend.figure_format = 'retina'

print("✓ Libraries imported")

In [ ]:
# Load integrated results
data_path = Path('../output/4_groups/integrated_results_cortex.csv')

if not data_path.exists():
    print(f"❌ File not found: {data_path}")
    print("Please run the analysis pipeline first to generate integrated results.")
else:
    df = pd.read_csv(data_path)
    print(f"✓ Loaded {len(df)} metabolites")
    print(f"\nColumns: {list(df.columns)}")
    display(df.head())

## 2. Data Exploration

In [ ]:
# Check available comparisons
fc_cols = [col for col in df.columns if col.startswith('log2FC_')]
print("Available comparisons:")
for col in fc_cols:
    print(f"  - {col}")

# Summary statistics
print(f"\nSignificant metabolites: {df['significant'].sum()} / {len(df)}")
print(f"FDR < 0.05: {(df['p_adj'] < 0.05).sum()}")

## 3. Volcano Plot Function

In [ ]:
def create_volcano_plot(
    df,
    comparison='glyoxylate_vs_saline',
    fdr_threshold=0.05,
    fc_threshold=1.5,
    figsize=(10, 8),
    point_size=50,
    alpha=0.6,
    colors=None,
    title=None
):
    """
    Create volcano plot for differential metabolite analysis.
    
    Parameters:
    -----------
    df : DataFrame
        Integrated results with log2FC and p_adj columns
    comparison : str
        Comparison name (e.g., 'glyoxylate_vs_saline')
    fdr_threshold : float
        FDR significance threshold (default: 0.05)
    fc_threshold : float
        Fold change threshold (default: 1.5, i.e., log2FC = ±0.585)
    figsize : tuple
        Figure size (width, height)
    point_size : int
        Scatter point size
    alpha : float
        Point transparency (0-1)
    colors : dict
        Custom colors for {'up', 'down', 'ns'}
    title : str
        Custom plot title
    
    Returns:
    --------
    fig, ax : matplotlib figure and axis
    """
    
    # Get fold change column
    fc_col = f'log2FC_{comparison}'
    if fc_col not in df.columns:
        raise ValueError(f"Column {fc_col} not found. Available: {[c for c in df.columns if 'log2FC' in c]}")
    
    # Prepare data
    plot_df = df[[fc_col, 'p_adj', 'm_z_bin']].copy()
    plot_df['neg_log10_padj'] = -np.log10(plot_df['p_adj'])
    
    # Classify points
    log2_fc_threshold = np.log2(fc_threshold)
    
    plot_df['category'] = 'ns'  # not significant
    plot_df.loc[
        (plot_df['p_adj'] < fdr_threshold) & (plot_df[fc_col] > log2_fc_threshold),
        'category'
    ] = 'up'
    plot_df.loc[
        (plot_df['p_adj'] < fdr_threshold) & (plot_df[fc_col] < -log2_fc_threshold),
        'category'
    ] = 'down'
    
    # Default colors
    if colors is None:
        colors = {
            'up': '#d62728',      # red
            'down': '#1f77b4',    # blue
            'ns': '#7f7f7f'       # gray
        }
    
    # Create plot
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot each category
    for category, color in colors.items():
        data = plot_df[plot_df['category'] == category]
        ax.scatter(
            data[fc_col],
            data['neg_log10_padj'],
            c=color,
            s=point_size,
            alpha=alpha,
            label=f"{category.upper()} ({len(data)})",
            edgecolors='none'
        )
    
    # Add threshold lines
    ax.axhline(
        -np.log10(fdr_threshold),
        color='black',
        linestyle='--',
        linewidth=1,
        alpha=0.5,
        label=f'FDR = {fdr_threshold}'
    )
    ax.axvline(
        log2_fc_threshold,
        color='black',
        linestyle='--',
        linewidth=1,
        alpha=0.5
    )
    ax.axvline(
        -log2_fc_threshold,
        color='black',
        linestyle='--',
        linewidth=1,
        alpha=0.5,
        label=f'FC = ±{fc_threshold}'
    )
    
    # Labels and title
    ax.set_xlabel('log₂ Fold Change', fontsize=14, fontweight='bold')
    ax.set_ylabel('-log₁₀ (FDR)', fontsize=14, fontweight='bold')
    
    if title is None:
        title = f'Volcano Plot: {comparison.replace("_", " ").title()}'
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Legend
    ax.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)
    
    # Grid
    ax.grid(True, alpha=0.3, linestyle=':')
    
    # Tight layout
    plt.tight_layout()
    
    return fig, ax

## 4. Create Volcano Plot

In [ ]:
# Basic volcano plot
fig, ax = create_volcano_plot(
    df,
    comparison='glyoxylate_vs_saline',
    fdr_threshold=0.05,
    fc_threshold=1.5
)
plt.show()

## 5. Customize Parameters

Experiment with different parameters to find the best visualization.

In [ ]:
# Try different thresholds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stricter threshold
plt.sca(axes[0])
create_volcano_plot(
    df,
    comparison='glyoxylate_vs_saline',
    fdr_threshold=0.01,
    fc_threshold=2.0,
    title='Strict Thresholds (FDR<0.01, FC>2)'
)

# Lenient threshold
plt.sca(axes[1])
create_volcano_plot(
    df,
    comparison='glyoxylate_vs_saline',
    fdr_threshold=0.1,
    fc_threshold=1.2,
    title='Lenient Thresholds (FDR<0.1, FC>1.2)'
)

plt.tight_layout()
plt.show()

In [ ]:
# Custom colors
custom_colors = {
    'up': '#e74c3c',      # vibrant red
    'down': '#3498db',    # vibrant blue
    'ns': '#95a5a6'       # light gray
}

fig, ax = create_volcano_plot(
    df,
    comparison='glyoxylate_vs_saline',
    colors=custom_colors,
    point_size=80,
    alpha=0.7
)
plt.show()

## 6. Multiple Comparisons

In [ ]:
# Get all comparisons
comparisons = [col.replace('log2FC_', '') for col in df.columns if col.startswith('log2FC_')]

# Create subplot for each comparison
n_comparisons = len(comparisons)
fig, axes = plt.subplots(1, n_comparisons, figsize=(8*n_comparisons, 6))

if n_comparisons == 1:
    axes = [axes]

for i, comparison in enumerate(comparisons):
    plt.sca(axes[i])
    create_volcano_plot(
        df,
        comparison=comparison,
        fdr_threshold=0.05,
        fc_threshold=1.5
    )

plt.tight_layout()
plt.show()

## 7. Label Significant Points

In [ ]:
# Create plot with labels for top significant metabolites
fig, ax = create_volcano_plot(
    df,
    comparison='glyoxylate_vs_saline',
    fdr_threshold=0.05,
    fc_threshold=1.5
)

# Get top significant metabolites
fc_col = 'log2FC_glyoxylate_vs_saline'
sig_df = df[(df['p_adj'] < 0.05) & (abs(df[fc_col]) > np.log2(1.5))].copy()
sig_df['abs_fc'] = abs(sig_df[fc_col])
top_sig = sig_df.nlargest(5, 'abs_fc')

# Add labels
for _, row in top_sig.iterrows():
    ax.annotate(
        row['m_z_bin'],
        xy=(row[fc_col], -np.log10(row['p_adj'])),
        xytext=(10, 10),
        textcoords='offset points',
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', lw=1)
    )

plt.show()

## 8. Export Publication-Ready Figure

In [ ]:
# Create final plot
fig, ax = create_volcano_plot(
    df,
    comparison='glyoxylate_vs_saline',
    fdr_threshold=0.05,
    fc_threshold=1.5,
    figsize=(10, 8),
    point_size=60,
    alpha=0.7
)

# Save as high-resolution PNG
output_dir = Path('../output/4_groups/plots')
output_dir.mkdir(exist_ok=True)

output_path = output_dir / 'volcano_plot_glyoxylate_vs_saline.png'
fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✓ Saved to: {output_path}")

# Also save as PDF for publications
output_path_pdf = output_dir / 'volcano_plot_glyoxylate_vs_saline.pdf'
fig.savefig(output_path_pdf, bbox_inches='tight', facecolor='white')
print(f"✓ Saved to: {output_path_pdf}")

plt.show()

## 9. Summary Statistics

In [ ]:
# Print summary
fc_col = 'log2FC_glyoxylate_vs_saline'
fdr_threshold = 0.05
fc_threshold = 1.5
log2_fc_threshold = np.log2(fc_threshold)

total = len(df)
significant = (df['p_adj'] < fdr_threshold).sum()
upregulated = ((df['p_adj'] < fdr_threshold) & (df[fc_col] > log2_fc_threshold)).sum()
downregulated = ((df['p_adj'] < fdr_threshold) & (df[fc_col] < -log2_fc_threshold)).sum()

print("="*50)
print("VOLCANO PLOT SUMMARY")
print("="*50)
print(f"Total metabolites: {total}")
print(f"Significant (FDR < {fdr_threshold}): {significant} ({significant/total*100:.1f}%)")
print(f"Upregulated (FC > {fc_threshold}): {upregulated}")
print(f"Downregulated (FC < {1/fc_threshold:.2f}): {downregulated}")
print("="*50)

# Show top upregulated
print("\nTop 5 Upregulated:")
up_df = df[(df['p_adj'] < fdr_threshold) & (df[fc_col] > log2_fc_threshold)]
if len(up_df) > 0:
    display(up_df.nlargest(5, fc_col)[['m_z_bin', fc_col, 'p_adj']])
else:
    print("None")

# Show top downregulated
print("\nTop 5 Downregulated:")
down_df = df[(df['p_adj'] < fdr_threshold) & (df[fc_col] < -log2_fc_threshold)]
if len(down_df) > 0:
    display(down_df.nsmallest(5, fc_col)[['m_z_bin', fc_col, 'p_adj']])
else:
    print("None")